In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# 1. Generate Synthetic Dataset
np.random.seed(42)
n_samples = 25
names = ["James Smith", "Maria Garcia", "Robert Johnson", "David Miller", "Linda Davis",
         "Michael Rodriguez", "Mary Martinez", "William Hernandez", "Barbara Lopez", "Richard Gonzalez",
         "Joseph Wilson", "Susan Anderson", "Thomas Thomas", "Jessica Taylor", "Charles Moore",
         "Sarah Jackson", "Christopher Martin", "Karen Lee", "Daniel Perez", "Nancy Thompson",
         "Lisa White", "Kevin Harris", "Edward Clark", "Betty Lewis", "Steven Walker"]

debt_age = np.random.randint(15, 180, n_samples)
amount_owed = np.round(np.random.uniform(500, 15000, n_samples), 2)
past_behavior = np.random.uniform(0.1, 0.9, n_samples)

# Rule-based logic for synthetic "Target" data
def assign_channel(age, amount):
    if amount > 10000 or age > 120: return 'Human Agent'
    if amount > 5000: return 'Voice Bot'
    if age < 45: return 'SMS'
    return 'Email'

def assign_time(age):
    if age < 60: return 'Evening'
    if age < 120: return 'Afternoon'
    return 'Morning'

channels = [assign_channel(a, m) for a, m in zip(debt_age, amount_owed)]
times = [assign_time(a) for a in debt_age]

df = pd.DataFrame({
    'Name': names, 'Debt Age (Days)': debt_age, 
    'Amount Owed': amount_owed, 'Past Behavior': past_behavior,
    'Recommended Channel': channels, 'Best Time': times
})

# 2. Train Model to Predict Best Channel
le = LabelEncoder()
df['Channel_Enc'] = le.fit_transform(df['Recommended Channel'])
X = df[['Debt Age (Days)', 'Amount Owed', 'Past Behavior']]
y = df['Channel_Enc']

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X, y)

# 3. Calculate Probability of Payment
# Formula: P = 1 / (1 + e^-logit)
logit = (3 * past_behavior) - (0.015 * debt_age) - (0.00005 * amount_owed) + 1
df['Prob. of Payment'] = np.round(1 / (1 + np.exp(-logit)), 4)

# 4. Generate Report
report = df[['Name', 'Debt Age (Days)', 'Amount Owed', 'Recommended Channel', 'Best Time', 'Prob. of Payment']]
print(report.to_string(index=False))

              Name  Debt Age (Days)  Amount Owed Recommended Channel Best Time  Prob. of Payment
       James Smith              117      8108.97           Voice Bot Afternoon            0.3148
      Maria Garcia              107      6763.20           Voice Bot Afternoon            0.8233
    Robert Johnson               29      4722.82                 SMS   Evening            0.7773
      David Miller              121      9371.87         Human Agent   Morning            0.6471
       Linda Davis               86      2522.66               Email Afternoon            0.6529
 Michael Rodriguez               35      4736.10                 SMS   Evening            0.8565
     Mary Martinez              117      5812.25           Voice Bot Afternoon            0.6380
 William Hernandez              136      7113.01         Human Agent   Morning            0.3425
     Barbara Lopez               89     11885.05         Human Agent Afternoon            0.8452
  Richard Gonzalez            